In [1]:
import pandas as pd
import numpy as np
import os
import time

# DATASET EXPLORATION AND EDA
here I explore the data set, it's structure and perform eda.

In [2]:
s=""
with open("/kaggle/input/datasets/melikechan/cifar100/cifar100/label_names.txt","r") as f:
    for line in f:
        s+=line
all_labels=s.strip().split('\n')

In [3]:
#seeing what files there are

total={}
for j in all_labels:
    x=os.listdir(f"/kaggle/input/datasets/melikechan/cifar100/cifar100/train/{j}")
    for i in x:
        full=os.path.splitext(f"/kaggle/input/datasets/melikechan/cifar100/cifar100/train/{j}/{i}")[1]
        total[full]=total.get(full,0)+1
print(total)

{'.png': 50000}


# TRANSFORMATIONS AND SETUP

In [4]:
import torchvision as tv
from torchvision import transforms as tr

In [5]:
m=(0.5071, 0.4867, 0.4408)
s=(0.2675, 0.2565, 0.2761)
traint=tr.Compose(
    [
        tr.RandomCrop(32,padding=4), #because cifar are 32x32 images 
        tr.RandomHorizontalFlip(),
        tr.Resize((224,224)),
        tr.ToTensor(),
        tr.Normalize(mean=m,std=s)
    ]
)
testt=tr.Compose(
    [
        tr.Resize((224,224)),
        tr.ToTensor(),
        tr.Normalize(mean=m,std=s)
    ]
)

In [6]:
traind=tv.datasets.ImageFolder("/kaggle/input/datasets/melikechan/cifar100/cifar100/train",transform=traint)
testd=tv.datasets.ImageFolder("/kaggle/input/datasets/melikechan/cifar100/cifar100/test",transform=testt)

In [7]:
x,y=traind[0]
print("Checking all labels and class labels are same: ",end="")
print(traind.classes==all_labels)
print("Length of train",len(traind))
print("Length of test",len(testd))

Checking all labels and class labels are same: True
Length of train 50000
Length of test 10000


## breaking down the dataset into parts

In [8]:
import random
import torch

In [9]:
random.seed(67) #for reproduce
ind=list(range(len(traind)))
random.shuffle(ind)
print("So here we take all the indices from 0 to 50000 and then we randomly shuffle it\n"
      +"we will then take 10% 25% 50% and 100% of the data to analyse the performance")

So here we take all the indices from 0 to 50000 and then we randomly shuffle it
we will then take 10% 25% 50% and 100% of the data to analyse the performance


In [10]:
d10=torch.utils.data.Subset(traind,ind[:5000]) # 10% data
d25=torch.utils.data.Subset(traind,ind[:12500]) # 25% data
d50=torch.utils.data.Subset(traind,ind[:25000]) # 50% data
d100=torch.utils.data.Subset(traind,ind) #all data

## DATA LOADER

In [11]:
from torch.utils.data import DataLoader as dl

train_loader_d10=dl(d10,batch_size=64,shuffle=True)
train_loader_d25=dl(d25,batch_size=64,shuffle=True)
train_loader_d50=dl(d50,batch_size=64,shuffle=True)
train_loader_full=dl(d100,batch_size=64,shuffle=True)

test_loader=dl(testd,batch_size=64,shuffle=False)

In [12]:
#inspection
images, labels = next(iter(train_loader_d10))
print("Images shape is",images.shape)
print("Labels shape is",labels.shape)

Images shape is torch.Size([64, 3, 224, 224])
Labels shape is torch.Size([64])


# MODEL INITIALIZATION

In [13]:
import torch as tor
import timm as tim
model=tim.create_model("vit_tiny_patch16_224", pretrained=False,num_classes=100)
dev=tor.device("cuda")

In [14]:
images, labels = next(iter(train_loader_d10))
print(images.shape)

print(model.patch_embed.img_size)
model=model.to(dev)

torch.Size([64, 3, 224, 224])
(224, 224)


In [15]:
images=images.to(dev)
outputs=model(images)
print(outputs.shape) #64 per batch and 100 classes
putputs=outputs.to(dev)
labels=labels.to(dev)

torch.Size([64, 100])


In [16]:
criterion=tor.nn.CrossEntropyLoss()
loss=criterion(outputs,labels)
loss.item()

4.709283828735352

In [17]:
optimizer=tor.optim.AdamW(model.parameters(),lr=0.0003)

In [18]:
model.train()
lossrun=0
for i,l in train_loader_d10:
    i=i.to(dev)
    l=l.to(dev)
    optimizer.zero_grad()
    outputs=model(i)
    loss=criterion(outputs,l)
    loss.backward()
    optimizer.step()
    lossrun+=loss.item()


In [19]:
print(lossrun/len(train_loader_d10))

4.423152712327015


In [20]:
model.eval()
correct=0
total=0
with tor.no_grad():
    for images,labels in test_loader:
        images=images.to(dev)
        labels=labels.to(dev)
        outputs=model(images)
        preds=outputs.argmax(dim=1)
        correct+=(preds==labels).sum().item() #.item to convert tensor to int
        total+=labels.size(0) 


In [21]:
print(f"Correct = {correct}\nTotal = {total}")
print("Accuracy = ",correct/total)

Correct = 418
Total = 10000
Accuracy =  0.0418


# MODEL TRAINING

In [22]:
tepoch=10

In [23]:
params = sum(p.numel() for p in model.parameters())
params_m=params/1000000 #parameter in millions 

## 10% data

In [24]:
start=time.time()

In [25]:
for epoch in range(tepoch):
    model.train()
    loss_s=0
    for i,l in train_loader_d10:
        i=i.to(dev)
        l=l.to(dev)
        optimizer.zero_grad()
        outputs = model(i)
        loss=criterion(outputs,l)
        loss.backward()
        optimizer.step()
        loss_s+=loss.item()
    print(f"Epoch {epoch} training done")
    print("Average loss across batch =",loss_s/len(train_loader_d10))
    model.eval()
    cor=0
    tot=0
    with torch.no_grad():
        for i,l in test_loader:
            i=i.to(dev)
            l=l.to(dev)
            outputs=model(i)
            preds=outputs.argmax(dim=1)
            cor+=(l==preds).sum().item()
            tot+=l.size(0)
        acc=cor/tot
    print(f"Epoch {epoch} evaluation done\nAccuracy = {acc}\n")

Epoch 0 training done
Average loss across batch = 4.245482604714889
Epoch 0 evaluation done
Accuracy = 0.0527

Epoch 1 training done
Average loss across batch = 4.136148751536502
Epoch 1 evaluation done
Accuracy = 0.069

Epoch 2 training done
Average loss across batch = 4.0258332717267775
Epoch 2 evaluation done
Accuracy = 0.076

Epoch 3 training done
Average loss across batch = 3.9243178971206087
Epoch 3 evaluation done
Accuracy = 0.09

Epoch 4 training done
Average loss across batch = 3.8315536191191852
Epoch 4 evaluation done
Accuracy = 0.0984

Epoch 5 training done
Average loss across batch = 3.7549319810505155
Epoch 5 evaluation done
Accuracy = 0.0994

Epoch 6 training done
Average loss across batch = 3.6889282178275193
Epoch 6 evaluation done
Accuracy = 0.1154

Epoch 7 training done
Average loss across batch = 3.644140717349475
Epoch 7 evaluation done
Accuracy = 0.1237

Epoch 8 training done
Average loss across batch = 3.550587509251848
Epoch 8 evaluation done
Accuracy = 0.1235



In [26]:
end=time.time()
train_time=end-start
print(f"Time take for training = {train_time}")

Time take for training = 684.5282554626465


In [27]:
print(f"Efficiency is {acc/params_m}")

Efficiency is 0.024279743046000193


In [28]:
torch.save(
    model.state_dict(),
    "vit_10pct.pth"
)

## 25% data

In [29]:
model=tim.create_model("vit_tiny_patch16_224", pretrained=False,num_classes=100)
optimizer=tor.optim.AdamW(model.parameters(),lr=0.0003)
model=model.to(dev)

In [30]:
start=time.time()

In [31]:
for epoch in range(tepoch):
    model.train()
    loss_s=0
    for i,l in train_loader_d25:
        i=i.to(dev)
        l=l.to(dev)
        optimizer.zero_grad()
        outputs = model(i)
        loss=criterion(outputs,l)
        loss.backward()
        optimizer.step()
        loss_s+=loss.item()
    print(f"Epoch {epoch} training done")
    print("Average loss across batch =",loss_s/len(train_loader_d25))
    model.eval()
    cor=0
    tot=0
    with torch.no_grad():
        for i,l in test_loader:
            i=i.to(dev)
            l=l.to(dev)
            outputs=model(i)
            preds=outputs.argmax(dim=1)
            cor+=(l==preds).sum().item()
            tot+=l.size(0)
        acc=cor/tot
    print(f"Epoch {epoch} evaluation done\nAccuracy = {acc}\n")
    
    

Epoch 0 training done
Average loss across batch = 4.307753964346283
Epoch 0 evaluation done
Accuracy = 0.0614

Epoch 1 training done
Average loss across batch = 4.008823211095771
Epoch 1 evaluation done
Accuracy = 0.0893

Epoch 2 training done
Average loss across batch = 3.808400723398948
Epoch 2 evaluation done
Accuracy = 0.1161

Epoch 3 training done
Average loss across batch = 3.6724855048315868
Epoch 3 evaluation done
Accuracy = 0.1291

Epoch 4 training done
Average loss across batch = 3.5521770302130253
Epoch 4 evaluation done
Accuracy = 0.1428

Epoch 5 training done
Average loss across batch = 3.4429413846560886
Epoch 5 evaluation done
Accuracy = 0.1462

Epoch 6 training done
Average loss across batch = 3.3622586544679134
Epoch 6 evaluation done
Accuracy = 0.1679

Epoch 7 training done
Average loss across batch = 3.281292155080912
Epoch 7 evaluation done
Accuracy = 0.1763

Epoch 8 training done
Average loss across batch = 3.1984780929526506
Epoch 8 evaluation done
Accuracy = 0.18

In [32]:
end=time.time()
train_time=end-start
print(f"Time take for training = {train_time}")

Time take for training = 1173.1915583610535


In [33]:
print(f"Efficiency is {acc/params_m}")

Efficiency is 0.03479615478137769


In [34]:
torch.save(
    model.state_dict(),
    "vit_25pct.pth"
)

## 50% data

In [35]:
model=tim.create_model("vit_tiny_patch16_224", pretrained=False,num_classes=100)
model=model.to(dev)
optimizer=tor.optim.AdamW(model.parameters(),lr=0.0003)

In [36]:
start=time.time()

In [37]:
for epoch in range(tepoch):
    model.train()
    loss_s=0
    for i,l in train_loader_d50:
        i=i.to(dev)
        l=l.to(dev)
        optimizer.zero_grad()
        outputs = model(i)
        loss=criterion(outputs,l)
        loss.backward()
        optimizer.step()
        loss_s+=loss.item()
    print(f"Epoch {epoch} training done")
    print("Average loss across batch =",loss_s/len(train_loader_d50))
    model.eval()
    cor=0
    tot=0
    with torch.no_grad():
        for i,l in test_loader:
            i=i.to(dev)
            l=l.to(dev)
            outputs=model(i)
            preds=outputs.argmax(dim=1)
            cor+=(l==preds).sum().item()
            tot+=l.size(0)
        acc=cor/tot
    print(f"Epoch {epoch} evaluation done\nAccuracy = {acc}\n")
    
    

Epoch 0 training done
Average loss across batch = 4.167573452605616
Epoch 0 evaluation done
Accuracy = 0.0894

Epoch 1 training done
Average loss across batch = 3.755877038706904
Epoch 1 evaluation done
Accuracy = 0.1417

Epoch 2 training done
Average loss across batch = 3.5175678498299834
Epoch 2 evaluation done
Accuracy = 0.1665

Epoch 3 training done
Average loss across batch = 3.3510685704858103
Epoch 3 evaluation done
Accuracy = 0.1834

Epoch 4 training done
Average loss across batch = 3.2105481088009027
Epoch 4 evaluation done
Accuracy = 0.2004

Epoch 5 training done
Average loss across batch = 3.1060437771975233
Epoch 5 evaluation done
Accuracy = 0.2284

Epoch 6 training done
Average loss across batch = 2.981522557680564
Epoch 6 evaluation done
Accuracy = 0.2392

Epoch 7 training done
Average loss across batch = 2.8835218873475212
Epoch 7 evaluation done
Accuracy = 0.2495

Epoch 8 training done
Average loss across batch = 2.79046557138643
Epoch 8 evaluation done
Accuracy = 0.271

In [38]:
end=time.time()
train_time=end-start
print(f"Time take for training = {train_time}")

Time take for training = 1957.9650056362152


In [39]:
print(f"Efficiency is {acc/params_m}")

Efficiency is 0.05036333030046994


In [40]:
torch.save(
    model.state_dict(),
    "vit_50pct.pth"
)

## all data

In [41]:
model=tim.create_model("vit_tiny_patch16_224", pretrained=False,num_classes=100)
model=model.to(dev)
optimizer=tor.optim.AdamW(model.parameters(),lr=0.0003)

In [42]:
start=time.time()

In [43]:
for epoch in range(tepoch):
    model.train()
    loss_s=0
    for i,l in train_loader_full:
        i=i.to(dev)
        l=l.to(dev)
        optimizer.zero_grad()
        outputs = model(i)
        loss=criterion(outputs,l)
        loss.backward()
        optimizer.step()
        loss_s+=loss.item()
    print(f"Epoch {epoch} training done")
    print("Average loss across batch =",loss_s/len(train_loader_full))
    model.eval()
    cor=0
    tot=0
    with torch.no_grad():
        for i,l in test_loader:
            i=i.to(dev)
            l=l.to(dev)
            outputs=model(i)
            preds=outputs.argmax(dim=1)
            cor+=(l==preds).sum().item()
            tot+=l.size(0)
        acc=cor/tot
    print(f"Epoch {epoch} evaluation done\nAccuracy = {acc}\n")
    

Epoch 0 training done
Average loss across batch = 3.9807483619436277
Epoch 0 evaluation done
Accuracy = 0.1334

Epoch 1 training done
Average loss across batch = 3.4923489663911904
Epoch 1 evaluation done
Accuracy = 0.1731

Epoch 2 training done
Average loss across batch = 3.228702048206573
Epoch 2 evaluation done
Accuracy = 0.221

Epoch 3 training done
Average loss across batch = 3.0124343734263155
Epoch 3 evaluation done
Accuracy = 0.2621

Epoch 4 training done
Average loss across batch = 2.838880360278937
Epoch 4 evaluation done
Accuracy = 0.2848

Epoch 5 training done
Average loss across batch = 2.6756625856889786
Epoch 5 evaluation done
Accuracy = 0.3127

Epoch 6 training done
Average loss across batch = 2.527339631791615
Epoch 6 evaluation done
Accuracy = 0.3432

Epoch 7 training done
Average loss across batch = 2.399184120128222
Epoch 7 evaluation done
Accuracy = 0.3523

Epoch 8 training done
Average loss across batch = 2.2773199156117254
Epoch 8 evaluation done
Accuracy = 0.390

In [44]:
end=time.time()
train_time=end-start
print(f"Time take for training = {train_time}")

Time take for training = 3701.8802304267883


In [45]:
print(f"Efficiency is {acc/params_m}")

Efficiency is 0.07141419221330962


In [46]:
torch.save(
    model.state_dict(),
    "vit_100pct.pth"
)